In [1]:
import pandas as pd

data = {
    "area": [1000, 1200, 1500, 1800, 2000, 2200, 2500, 2800, 3000, 3200,
             1100, 1400, 1700, 1900, 2100, 2400, 2700, 2900, 3100, 3500],

    "bedrooms": [2, 2, 3, 3, 4, 4, 4, 5, 5, 5,
                 2, 3, 3, 4, 4, 4, 5, 5, 5, 6],

    "age": [20, 15, 10, 8, 5, 4, 3, 2, 1, 1,
            18, 12, 9, 7, 6, 4, 3, 2, 1, 1],

    "price": [150000, 180000, 230000, 270000, 320000,
              350000, 390000, 440000, 480000, 510000,
              165000, 215000, 250000, 300000, 330000,
              370000, 420000, 460000, 500000, 570000]
}

df = pd.DataFrame(data)

print(df.head())
print(df.shape)

   area  bedrooms  age   price
0  1000         2   20  150000
1  1200         2   15  180000
2  1500         3   10  230000
3  1800         3    8  270000
4  2000         4    5  320000
(20, 4)


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   area      20 non-null     int64
 1   bedrooms  20 non-null     int64
 2   age       20 non-null     int64
 3   price     20 non-null     int64
dtypes: int64(4)
memory usage: 768.0 bytes


In [3]:
X=df[["area","bedrooms","age"]]
y=df[["price"]].values.astype(int)

In [22]:
import numpy as np
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Scale features and target for stable training
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

.ravel() means convert a 2D array into a 1D array.

In [24]:
#convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train,dtype=torch.float32)
y_test = torch.tensor(y_test,dtype=torch.float32)

In [25]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self,features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self,idx):
        return self.features[idx], self.targets[idx]

In [26]:
#now train and loader dataset with test also
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [27]:
X_batch,y_batch=next(iter(train_loader))
print(X_batch.shape)
print(y_batch.shape)

torch.Size([4, 8])
torch.Size([4])


In [28]:
#now a simple regression model with 3 input features and 1 output feature
class SimpleRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 1)  # 3 input features, 1 output feature

    def forward(self,x):
        return self.fc1(x)
    
model=SimpleRegressionModel()

In [29]:
criterion=nn.MSELoss()
learning_rate=0.01

In [30]:
optimizer=optim.Adam(model.parameters(),lr=learning_rate)

num_epochs=100

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)
    loss= criterion(outputs.squeeze(), y_train)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

RuntimeError: mat1 and mat2 shapes cannot be multiplied (16512x8 and 3x1)

In [ ]:
# Evaluate on the test set
model.eval()
with torch.no_grad():
    test_predictions = model(X_test).squeeze(1)
    test_loss = criterion(test_predictions, y_test)

print(f"Final Test MSE (scaled target): {test_loss.item():.6f}")

# Convert predictions and targets back to original scale
original_predictions = scaler_y.inverse_transform(test_predictions.unsqueeze(1).numpy())
original_targets = scaler_y.inverse_transform(y_test.unsqueeze(1).numpy())

print("Example predictions vs actual prices:")
for pred, true in zip(original_predictions[:5], original_targets[:5]):
    print(f"Predicted: {pred[0]:.0f} | Actual: {true[0]:.0f}")

Final Test MSE (scaled target): 0.012315
Example predictions vs actual prices:
Predicted: 133728 | Actual: 150000
Predicted: 462688 | Actual: 460000
Predicted: 371404 | Actual: 370000
Predicted: 160396 | Actual: 180000


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Evaluate the trained model
model.eval()
with torch.no_grad():
    test_predictions = model(X_test).squeeze(1)
    test_loss = criterion(test_predictions, y_test)

# Convert scaled predictions/targets back to original price scale
test_predictions_original = scaler_y.inverse_transform(test_predictions.unsqueeze(1).numpy())
y_test_original = scaler_y.inverse_transform(y_test.unsqueeze(1).numpy())

# Compute regression metrics
mae = mean_absolute_error(y_test_original, test_predictions_original)
r2 = r2_score(y_test_original, test_predictions_original)

print(f"Test MSE (scaled): {test_loss.item():.6f}")
print(f"Test MAE (original scale): {mae:.2f}")
print(f"Test R^2 (original scale): {r2:.4f}")

# Show a few sample predictions
print("Sample predictions vs actual prices:")
for pred, true in zip(test_predictions_original[:5], y_test_original[:5]):
    print(f"Predicted: {pred[0]:.0f} | Actual: {true[0]:.0f}")

Test MSE (scaled): 0.012315
Test MAE (original scale): 9992.06
Test R^2 (original scale): 0.9902
Sample predictions vs actual prices:
Predicted: 133728 | Actual: 150000
Predicted: 462688 | Actual: 460000
Predicted: 371404 | Actual: 370000
Predicted: 160396 | Actual: 180000
